# Try SendSoon in Google Colab

Calls the same HTTP APIs as [`sendsoon/mcp`](https://github.com/sendsoon/mcp). Uses **`requests` only** (preinstalled in Colab — no `pip install`).

| MCP tool | Endpoint |
| --- | --- |
| `send_email` | `POST /api/send-test-email` |
| `ip_lookup` | `GET /api/ip/lookup` |
| `markitdown_convert` | `POST /api/markitdown/convert` |

Optional env (same as MCP): `SENDSOON_API_KEY`.

Run cells top to bottom: **Setup → send_email → ip_lookup → markitdown_convert**.

## 1. Setup

Set optional `SENDSOON_API_KEY` and recipient email. Leave recipient blank to skip the send_email test.

In [ ]:
import html
import json
import os
import uuid
from getpass import getpass

import requests

BASE = "https://www.sendsoonai.com"
API_KEY = os.environ.get("SENDSOON_API_KEY", "").strip() or None
recipient = input("Recipient for send_email (blank = skip): ").strip()
key_in = getpass("API Key (Enter = skip): ").strip()
if key_in:
    API_KEY = key_in

def headers(accept="application/json"):
    h = {"Accept": accept}
    if API_KEY:
        h["Authorization"] = f"Bearer {API_KEY}"
    return h

print("Base URL:", BASE)
print("API Key:", "set" if API_KEY else "anonymous trial")
print("Recipient:", recipient or "(skip send_email)")

## 2. Test `send_email`

Sends one test email when a recipient was entered in Setup. Without `SENDSOON_API_KEY`, one public IP gets up to 3 free sends per day.

In [ ]:
if not recipient:
    print("send_email: skipped (no recipient)")
else:
    body = "Configuration successful. Sent from Google Colab."
    r = requests.post(
        f"{BASE}/api/send-test-email",
        headers={**headers(), "Content-Type": "application/json", "Idempotency-Key": str(uuid.uuid4())},
        json={
            "to": recipient,
            "subject": "SendSoon Colab test",
            "htmlContent": f'<pre style="white-space:pre-wrap">{html.escape(body)}</pre>',
        },
        timeout=30,
    )
    print(f"send_email: HTTP {r.status_code}")
    print(r.text)

## 3. Test `ip_lookup`

Look up geolocation and ISP info for a public IP. No API Key required.

In [ ]:
r = requests.get(f"{BASE}/api/ip/lookup", params={"ip": "8.8.8.8"}, headers=headers(), timeout=30)
r.raise_for_status()
print("ip_lookup:", json.dumps(r.json(), indent=2, ensure_ascii=False))

## 4. Test `markitdown_convert`

Downloads [`SendSoon_Overview.docx`](https://github.com/sendsoon/mcp/blob/main/docs/SendSoon_Overview.docx) from GitHub and converts it to Markdown.

In [ ]:
DOCX_URL = "https://raw.githubusercontent.com/sendsoon/mcp/main/docs/SendSoon_Overview.docx"
DOCX_NAME = "SendSoon_Overview.docx"

print(f"Downloading: {DOCX_URL}")
docx_resp = requests.get(DOCX_URL, timeout=30)
docx_resp.raise_for_status()

r = requests.post(
    f"{BASE}/api/markitdown/convert",
    headers=headers(accept="text/markdown, application/json"),
    files={"file": (DOCX_NAME, docx_resp.content)},
    timeout=60,
)
r.raise_for_status()
if "application/json" in (r.headers.get("content-type") or ""):
    print("markitdown:", r.json().get("markdown", ""))
else:
    print("markitdown:", r.text)